# 🏞 Convolutional Neural Network

In this notebook, we'll walk through the steps required to train your own convolutional neural network (CNN) on the CIFAR dataset

In [ ]:
import numpy as np

import torch
from torch import nn
from torch.utils.data import Dataset
from torchvision import datasets
from transformers import Trainer, TrainingArguments

from notebooks.utils_pytorch import display

## 0. Parameters <a name="parameters"></a>

In [ ]:
NUM_CLASSES = 10

## 1. Prepare the Data <a name="prepare"></a>

In [ ]:
train_set = datasets.CIFAR10(root="../../../data", train=True, download=True)
test_set = datasets.CIFAR10(root="../../../data", train=False, download=True)

# .data is a uint8 NumPy array of shape (N, 32, 32, 3), .targets a list of class ids
(x_train, y_train) = (train_set.data, np.array(train_set.targets))
(x_test, y_test) = (test_set.data, np.array(test_set.targets))

In [ ]:
# Scale to [0, 1] and move channels first: (N, H, W, C) -> (N, C, H, W),
# which is the image layout PyTorch layers expect
x_train = torch.tensor(x_train, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0
x_test = torch.tensor(x_test, dtype=torch.float32).permute(0, 3, 1, 2) / 255.0

# No one-hot encoding needed: nn.CrossEntropyLoss takes integer class ids
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


# The Trainer feeds each example to the model as keyword arguments,
# so every example is a dict whose keys match model.forward()
class ImageDataset(Dataset):
    def __init__(self, images, labels):
        self.images, self.labels = images, labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return {"pixel_values": self.images[idx], "labels": self.labels[idx]}


train_dataset = ImageDataset(x_train, y_train)
test_dataset = ImageDataset(x_test, y_test)

In [ ]:
display(x_train[:10])
print(y_train[:10])

## 2. Build the model <a name="build"></a>

In [ ]:
def conv_block(in_channels, out_channels, stride):
    # padding=1 keeps the 3x3 conv "same"-sized (stride 2 halves the size).
    # Keras BatchNormalization(momentum=0.99, epsilon=1e-3) == PyTorch momentum=0.01,
    # and Keras' LeakyReLU uses a default slope of 0.3
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1),
        nn.BatchNorm2d(out_channels, momentum=0.01, eps=1e-3),
        nn.LeakyReLU(0.3),
    )


class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            conv_block(3, 32, stride=1),  # -> (32, 32, 32)
            conv_block(32, 32, stride=2),  # -> (32, 16, 16)
            conv_block(32, 64, stride=1),  # -> (64, 16, 16)
            conv_block(64, 64, stride=2),  # -> (64, 8, 8)
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.BatchNorm1d(128, momentum=0.01, eps=1e-3),
            nn.LeakyReLU(0.3),
            nn.Dropout(p=0.5),
            # No softmax here: the layer outputs raw scores (logits) and
            # nn.CrossEntropyLoss applies the softmax internally
            nn.Linear(128, num_classes),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    # The Trainer expects the model to compute its own loss when labels are given
    def forward(self, pixel_values, labels=None):
        logits = self.net(pixel_values)
        loss = self.loss_fn(logits, labels) if labels is not None else None
        return {"loss": loss, "logits": logits}


model = CNN(NUM_CLASSES)
print(model)
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

## 3. Train the model <a name="train"></a>

In [ ]:
# The Trainer picks the best available device (GPU, Apple MPS or CPU) automatically
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

# TrainingArguments holds the optimizer and training settings
# (the Trainer uses AdamW; with weight_decay=0 it is plain Adam)
training_args = TrainingArguments(
    output_dir="./checkpoint",
    learning_rate=0.0005,
    weight_decay=0.0,
    lr_scheduler_type="constant",  # Keras keeps the learning rate fixed
    per_device_train_batch_size=32,
    per_device_eval_batch_size=1000,
    num_train_epochs=10,
    eval_strategy="epoch",  # validate on the test set after every epoch
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (logits.argmax(axis=-1) == labels).mean()}


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,  # like Keras' validation_data
    compute_metrics=compute_metrics,
)

In [ ]:
# The Trainer runs the loop, the train()/eval() mode switches (BatchNorm,
# Dropout), the backward pass and the optimizer step for us
trainer.train()

## 4. Evaluation <a name="evaluate"></a>

In [ ]:
trainer.evaluate(test_dataset)

In [ ]:
CLASSES = np.array(
    [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]
)

logits = trainer.predict(test_dataset).predictions
preds = torch.tensor(logits).softmax(dim=1)
preds_single = CLASSES[preds.argmax(dim=1).numpy()]
actual_single = CLASSES[y_test.numpy()]

In [ ]:
import matplotlib.pyplot as plt

n_to_show = 10
indices = np.random.choice(range(len(x_test)), n_to_show)

fig = plt.figure(figsize=(15, 3))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img = x_test[idx].permute(1, 2, 0)  # (C, H, W) -> (H, W, C) for matplotlib
    ax = fig.add_subplot(1, n_to_show, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        "pred = " + str(preds_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.text(
        0.5,
        -0.7,
        "act = " + str(actual_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(img)